# ZeroER++ — full-data benchmarks (phase 2)

Takes each dataset's tuning Pareto front and re-evaluates it on the **full data**, into sibling `<dataset>_full` studies in the same `outputs/optuna.db`. Run `hyperparameter-tuning.ipynb` first.

## Setup

Generic runners/views in `utils`; the objective + its knobs are declared below and **must match the tuning run** (enqueued configs replay through the same distributions).

In [ ]:
from pyjedai_module import ZeroEREstimator
from utils import (DATASETS, DEFAULT_TUNE_FRACTION, load_data,
                   precision_recall_f1, blocking_recall,   # objective helpers
                   retrain_all, export, show_results)

## Search space

Same `SPACE`/`SEED` as tuning; no budget needed — phase 2 only enqueues the existing front.

In [ ]:
SEED = 0
SPACE = {                                     # the knobs; each key is a flat ZeroEREstimator arg
    # embeddings branch — PLM blocking. Static vectorizers (word2vec/glove/...) excluded: not LMs.
    "vectorizer": ["sminilm", "smpnet", "st5", "sdistilroberta",   # sentence-transformer PLMs
                   "bert", "distilbert", "roberta", "xlnet", "albert"],  # token PLMs
    "top_k": (1, 10),                         # neighbours kept per entity
    "similarity_distance": ["cosine", "euclidean"],
    # standard branch — syntactic blocking + cleaning
    "smoothing_factor": (1.0, 1.5),           # BlockPurging threshold
    "block_filtering_ratio": (0.5, 0.95),     # BlockFiltering keep-ratio
    "weighting_scheme": ["CBS", "ECBS", "JS", "EJS", "X2"],  # WeightedEdgePruning
    # matcher (searched in both branches)
    "c_bay": (1e-3, 3e-1),                    # ZeroER EM regularisation (log-float)
}

## Objective

The per-experiment trial — declared here, not in `utils`, so a new experiment is just a copy-and-tweak of this cell. It returns `(F1, candset, pair_completeness)` and carries its own `directions`; `make_objective` binds it to `SPACE`/`SEED` for the runners.

In [ ]:
class TuningObjective:
    """Multi-objective trial: (max F1, min surviving block pairs, max pair completeness).

    One ``Data`` is loaded per instance and reused across trials; ``get_params``
    *is* the search space (keys map one-to-one onto the ZeroEREstimator ctor).
    """
    directions = ["maximize", "minimize", "maximize"]   # F1 up, block pairs down, PC up

    def __init__(self, dataset, space, seed=0, partition=True):
        self.space = space
        self.source_trial = None              # set by retrain_full to stamp each full trial
        self.data = load_data(dataset, partition=partition, seed=seed)
        self.attributes = DATASETS[dataset]["attributes"]
        frac = DATASETS[dataset].get("tune_fraction", DEFAULT_TUNE_FRACTION)
        scope = f"partition (frac={frac})" if partition else "full data"
        print(f"    {scope}: {self.data.num_of_entities_1}+{self.data.num_of_entities_2}"
              f" entities, {len(self.data.ground_truth)} matches", flush=True)

    def get_params(self, trial):
        # conditional: the blocker is itself a knob; each branch exposes only its
        # own stage params; c_bay (the matcher) is searched in both branches.
        s = self.space
        blocker = trial.suggest_categorical("blocker", ["embeddings", "standard"])
        params = dict(blocker=blocker,
                      c_bay=trial.suggest_float("c_bay", *s["c_bay"], log=True))
        if blocker == "embeddings":
            params.update(
                vectorizer=trial.suggest_categorical("vectorizer", s["vectorizer"]),
                top_k=trial.suggest_int("top_k", *s["top_k"]),
                similarity_distance=trial.suggest_categorical("similarity_distance", s["similarity_distance"]),
            )
        else:
            params.update(
                smoothing_factor=trial.suggest_float("smoothing_factor", *s["smoothing_factor"]),
                block_filtering_ratio=trial.suggest_float("block_filtering_ratio", *s["block_filtering_ratio"]),
                weighting_scheme=trial.suggest_categorical("weighting_scheme", s["weighting_scheme"]),
            )
        return params

    def __call__(self, trial):
        est = ZeroEREstimator(attributes=self.attributes, **self.get_params(trial))
        graph = est.fit_predict(self.data)
        _, _, f1 = precision_recall_f1(graph, self.data)
        pair_completeness = blocking_recall(est.blocks, self.data)   # gold kept by the blocker
        trial.set_user_attr("candset_size", est.candset_size)
        trial.set_user_attr("blocking_seconds", round(est.blocking_seconds, 3))
        trial.set_user_attr("em_seconds", round(est.matcher.em_time, 3))
        if self.source_trial is not None:
            trial.set_user_attr("source_trial", self.source_trial)
        return f1, est.candset_size, pair_completeness


def make_objective(dataset, partition=True):                 # factory the runners call per dataset
    return TuningObjective(dataset, SPACE, SEED, partition=partition)

## Run phase 2 — retrain each front on the full data

Resumable: configs already retrained (stamped `source_trial`) are skipped.

In [ ]:
retrain_all(make_objective, seed=SEED)

## Results

This notebook produces the `full` fronts; the `partition` fronts are shown too when present in `optuna.db`.

In [ ]:
show_results()

## Export CSV views

Trial logs + the full-data Pareto front (`full_pareto_fronts.csv`) from `optuna.db`.

In [ ]:
export()